# RoadEye CityFlow Re-ID training
Run this in a private Google Colab GPU runtime. Upload the two ignored ZIP files to `MyDrive/RoadEye/`. They contain licensed CityFlow-derived crops and RoadEye code, so do not share them publicly. This notebook never uses S02/S04/S05 labels.

In [ ]:
import hashlib
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/RoadEye")
job_zip = drive_root / "roadeye-reid-colab-job.zip"
data_zip = drive_root / "roadeye-reid-training-data.zip"
assert (
    job_zip.is_file() and data_zip.is_file()
), "Upload both RoadEye ZIP files to MyDrive/RoadEye"
job_root, data_root = Path("/content/roadeye_job"), Path("/content/roadeye_data")
with zipfile.ZipFile(job_zip) as archive:
    archive.extractall(job_root)
with zipfile.ZipFile(data_zip) as archive:
    archive.extractall(data_root)
manifest = json.loads((job_root / "job-manifest.json").read_text())
for relative, expected_hash in manifest["source_sha256"].items():
    with (job_root / relative).open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == expected_hash

In [ ]:
# Colab supplies GPU-enabled torch. Record its versions; do not replace it with a CPU wheel.
import subprocess
import sys
import torch
import torchvision

assert torch.cuda.is_available(), "Select a GPU runtime before training"
print(
    sys.version,
    torch.__version__,
    torchvision.__version__,
    torch.cuda.get_device_name(),
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(job_root), "--no-deps"],
    check=True,
)

In [ ]:
# Explicit, pinned initialization download; training is the only network-dependent model step.
from urllib.request import urlretrieve

initial = Path("/content/veri_sbs_R50-ibn.pth")
url = (
    "https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth"
)
expected = "57fb9c17d88911ea64390bf5427f43511435e7f88f6eed9dbc969d4b611e53cd"
if not initial.is_file():
    urlretrieve(url, initial)
with initial.open("rb") as stream:
    actual = hashlib.file_digest(stream, "sha256").hexdigest()
assert actual == expected

In [ ]:
output = Path("/content/roadeye_result")
command = [
    sys.executable,
    str(job_root / "scripts/train_reid_portable.py"),
    "--bundle",
    str(data_root),
    "--config",
    str(job_root / "configs/reid-training.json"),
    "--initial-weights",
    str(initial),
    "--output",
    str(output),
]
subprocess.run(command, check=True)

In [ ]:
result_zip = shutil.make_archive("/content/roadeye-reid-result", "zip", output)
destination = shutil.copy2(result_zip, drive_root / "roadeye-reid-result.zip")
print("Saved result for local download:", destination)